This notebook contains the reusable functions required to create the ETL pipeline in `youtube_etl_pipeline`.

These functions were created in `exploration_notebook` through extensive EDA.

In [0]:
# Create a set of allowed regions
ALLOWED_REGIONS = {
    "CA", "DE", "FR", "GB", "IN",
    "JP", "KR", "MX", "RU", "US"
}

In [0]:
# Create the bronze ingestion function
def load_raw_csv(spark, path: str, region: str):
    
    # Validate region (fail fast)
    if region not in ALLOWED_REGIONS:
        raise ValueError(f"[INGESTION BLOCKED] Unknown region: {region}")

    # Read raw file
    df = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", "\"")
    .option("inferSchema", True)
    .csv(path)
    )

    # Add lineage column
    df = df.withColumn("region", lit(region))

    return df

In [0]:
def load_all_regions(spark, folder_path: str):
    
    dfs = []

    for file in os.listdir(folder_path):
        if file.endswith(".csv"):
            region = file.split("videos.csv")[0]
            print(f"Loading: {file} → region: {region}")  # ← add this

            df = load_raw_csv(
                spark,
                os.path.join(folder_path, file),
                region
            )

            dfs.append(df)

    print(f"Total DataFrames loaded: {len(dfs)}")  # ← and this

    if not dfs:
        raise ValueError(f"[INGESTION FAILED] No CSV files found in: {folder_path}")

    return reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

In [0]:
# After union, verify no rows were lost
def validate_bronze(df, expected_regions: set):

    print("=== BASIC VALIDATION ===")

    required_cols = ["video_id", "views", "likes", "region"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if "_corrupt_record" in df.columns:
        bad = df.filter(col("_corrupt_record").isNotNull()).count()
        print(f"Corrupt records: {bad}")

    print("Row count:", df.count())

    # Reconciliation: confirm all expected regions loaded
    loaded_regions = {r.region for r in df.select("region").distinct().collect()}
    missing_regions = expected_regions - loaded_regions
    if missing_regions:
        print(f"[WARNING] Missing regions after union: {missing_regions}")
    else:
        print(f"[OK] All {len(loaded_regions)} regions present")

    df.select("region").distinct().show()

    return df

In [0]:
# Bronze to silver transformation: cleaning and casting
# Expects raw string columns from bronze layer: do not call twice.
def transform_silver(df):
    return (
        df
        .withColumn("publish_time", to_timestamp("publish_time"))
        .withColumn("trending_date", to_date("trending_date", "yy.dd.MM"))
        .withColumn("views", col("views").cast("long"))
        .withColumn("likes", col("likes").cast("long"))
        .withColumn("dislikes", col("dislikes").cast("long"))
        .withColumn("comment_count", col("comment_count").cast("long"))
        .select(
            "video_id",
            "category_id",
            "title",
            "channel_title",
            "publish_time",
            "trending_date",
            "views",
            "likes",
            "dislikes",
            "comment_count",
            "comments_disabled",
            "ratings_disabled",
            "video_error_or_removed",
            "region"
        )
    )

In [0]:
# Load a single category JSON file
def load_raw_json(spark, file_path: str, region: str):

    df = (
        spark.read
             .option("multiline", "true")
             .json(file_path)
    )

    if df.limit(1).count() == 0:
        raise ValueError(f"[INGESTION FAILED] File is empty or unreadable: {file_path}")

    df = df.withColumn("region", lit(region))

    return df

In [0]:
# Create a batch loader for all category JSONs
def load_all_category_jsons(spark, folder_path: str):

    dfs = {}

    for file in os.listdir(folder_path):
        if file.endswith(".json"):
            region = file.split("_category_id")[0]

            df = load_raw_json(
                spark,
                os.path.join(folder_path, file),
                region
            )

            dfs[region] = df

    if not dfs:
        raise ValueError(f"[INGESTION FAILED] No JSON files found in: {folder_path}")

    return dfs

In [0]:
# Create function to validate raw category JSONs
def validate_category_raw(dfs: dict) -> dict:
    """
    Schema sanity check across all region DataFrames.
    Logs mismatches and row counts, then returns the dfs dict.
    """

    if not dfs:
        raise ValueError("[VALIDATION FAILED] Empty dataframe dictionary passed.")

    # Use simpleString() for reliable schema comparison
    base_region, base_df = next(iter(dfs.items()))
    base_schema_str = base_df.schema.simpleString()

    valid_dfs = {}

    for region, df in dfs.items():
        schema_str = df.schema.simpleString()

        if schema_str != base_schema_str:
            print(f"[WARNING] Schema mismatch in {region} vs {base_region} — skipping")
            print(f"  Expected: {base_schema_str}")
            print(f"  Got:      {schema_str}")
            continue

        row_count = df.count()
        print(f"[OK] {region}: rows={row_count}")
        valid_dfs[region] = df

    if not valid_dfs:
        raise ValueError("[VALIDATION FAILED] No DataFrames passed schema validation.")

    return valid_dfs  

In [0]:
def flatten_categories(df):
    """Explode items array and select relevant fields."""

    return (
        df
        .select(explode("items").alias("item"), "region")
        .select(
            col("item.id").alias("category_id"),
            col("item.snippet.title").alias("category_name"),
            "region"
        )
    )